# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Data Discovery & Debugging Trail (kept for transparency)

> **Note on process:** The taxonomy below wasn't designed in a vacuum. It emerged
> from a series of diagnostic passes on real data — each failed attempt taught us
> something about the data's actual structure that shaped the final design. These
> diagnostic cells are preserved as part of the engineering record.

### Critical finding: Namespace mismatch between datasets

`content_refresh_anonymized.csv` (the starter CSV from the research paper) and
`internship-warehouse` (the daily performance facts) use **completely different
anonymization namespaces**:

- Starter CSV IDs: `content_304f48230142...` (12-char hex suffix)
- Warehouse IDs: `content_945d6ff91386c817...` (16-char hex suffix)
- **Overlap: 0 pages** (verified in diagnostic cell below)

**Implication:** Any feature that depends on metadata from the starter CSV
(`word_count`, `days_since_last_update`) **cannot be merged** onto the warehouse
features. This is not a bug — it's a structural property of two separate datasets.

**Consequence for this notebook:** The `thin_with_potential` reason code (which
depended on `word_count < 1500`) was removed from the final taxonomy. Instead,
`content_age_days` was computed directly from `MIN(report_date)` in the warehouse
— a more honest source anyway (no survivor bias from the snapshot).

### What we learned through iteration

1. **First attempt** (5 reason codes from the design doc) → 94% `signal_unclear`
   — the taxonomy didn't match the data.
2. **Diagnostic pass** → Discovered zero overlap between starter CSV and warehouse.
   The metadata we expected wasn't available.
3. **Solution** → Computed `content_age_days` from `MIN(report_date)` in the daily
   panel.
4. **Final taxonomy** → Designed around what the RF model *actually* selects
   (engagement-rich pages, not zero-click pages), not what we assumed it would select.

This process — fail, diagnose, redesign — is the real product of this notebook.
The queue is the artifact; the reasoning is the evidence.

## 1. Ranked actions + reason codes

The queue: what to do first, and why, in words a human trusts.

> **Taxonomy note:** an earlier draft of this section listed five candidate reason
> codes (`stale_with_demand`, `declining_fast`, `position_slipping`,
> `engagement_gap`, `thin_with_potential`). That table is **superseded**: two of
> its codes depended on starter-CSV metadata that cannot be joined to the
> warehouse (Section 0), and the surviving taxonomy was rebuilt around what the
> model actually selects. **The operative taxonomy is the one implemented and
> printed in the code cells below** — see the final reason-code table in the
> design-decisions cell and the printed distribution in the final code cell.

### Design decisions

**The score is a ranking device, not a decision.** The RF probability orders pages by predicted recovery probability; it does not tell the specialist what to do. A separate deterministic layer maps observable signals to actionable reason codes.

**Why reason codes are rule-based, not SHAP-based:**
- The locked feature set includes instrumentation artifacts (`has_ga4_data`, `gsc_avg_position_is_placeholder`). SHAP on these would tell a human "because no GA4" — reading a measurement gap as a content problem.
- Global feature importance explains the model, not the page. Per-row SHAP answers "why did the model score high?" but the specialist needs "why does this page deserve my hour?"
- Rule-based codes live independently of model internals, surviving retraining.

**Why the taxonomy is fixed categories, not continuous ranges:**
- Capacity is discrete (the team reviews ~20–50 pages per week).
- Tools are discrete (refresh, rewrite, monitor).
- Auditability requires categorical dispositions for §4 measurement.

### Four categories (plus WATCH exception and NO_ACTION_LOGGED default)

| Tier | Population | Capacity |
|------|-----------|----------|
| `ACT_THIS_WEEK` | top 20 by RF score | reserved |
| `REVIEW_IF_CAPACITY` | ranks 21–50 | if bandwidth allows |
| `WATCH` | rank >50 AND `trend_30d ≤ -40%` AND `impressions ≥ demand_median` | zero capacity; glance list in weekly review |
| `NO_ACTION_LOGGED` | everything else | explicit silent state (not "leftover") |

### Reason codes (deterministic, priority order, first-fire = primary)

| Code | Condition | Plain-language reason |
|------|-----------|----------------------|
| `stale_visible` | age ≥ 180 days AND impressions ≥ median | "Old page, still earning traffic — refresh candidate" |
| `strong_engagement` | has_ga4_data AND sessions_organic ≥ 75th percentile | "Users engage deeply — protect from decay" |
| `high_position_traffic` | position < 5 AND impressions ≥ 75th percentile | "Top position, high visibility — expand opportunity" |
| `converting_visibility` | page-one AND clicks > 0 AND impressions ≥ median | "Ranking and converting — maintain" |
| `signal_unclear` | fallback (none of the above fired) | "Model flagged, observable signals don't explain" |

### Key empirical finding from the diagnostic

The RF model does **not** rank zero-click, high-visibility pages at the top (0 of 500 queued rows are zero-click). Instead it consistently selects GA4-tracked, engagement-rich pages. This matches Finding #4 from the ML-09 audit: refresh acts as a **stabilization brake** for already-valuable pages, not a recovery lever for broken ones. The rule (ML-07) and the model disagree by design — they're optimizing for different populations. The `cross_disagree` flag surfaces this disagreement for mandatory human review in §3.



> **Note:** The `thin_with_potential` code (word_count-based) was removed from
> this taxonomy due to the namespace mismatch documented in Section 0.

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [ ]:
# === w07 Section 1: Load matrix + compute scores ===
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive

# --- Mount Google Drive first so paths are accessible ---
drive.mount('/content/drive', force_remount=True)

# --- Load the clean feature matrix (already computed in w05/w06) ---
FILE_PATH = "/content/drive/MyDrive/FlyRankai-Internship/work/outputs/clean_features_label.parquet"
df = pd.read_parquet(FILE_PATH)
df['report_date'] = pd.to_datetime(df['report_date'])

print(f"Loaded {len(df):,} rows, date range: {df['report_date'].min()} to {df['report_date'].max()}")

# --- Define features (locked in w05) ---
FEATURES = [
    "gsc_clicks", "gsc_avg_position", "has_ga4_data",
    "ga4_engaged_sessions", "ga4_total_engagement_sec", "sessions_organic",
    "gsc_avg_position_is_placeholder"
]

# --- Preprocessing (same as w05/w06) ---
SKEWED_COLS = ["gsc_clicks", "ga4_engaged_sessions", "ga4_total_engagement_sec",
               "sessions_organic", "gsc_avg_position"]

def preprocess(X):
    X_proc = X[FEATURES].copy()
    for col in SKEWED_COLS:
        if col in X_proc.columns:
            X_proc[col] = np.log1p(X_proc[col].fillna(0))
    return X_proc

# --- Train RF on March 1-21, predict on March 22-31 (time-aware split from w06) ---
train_mask = df['report_date'] < '2026-03-22'
test_mask = df['report_date'] >= '2026-03-22'

X_train = preprocess(df[train_mask])
y_train = df[train_mask]['recovery_label']

X_test = preprocess(df[test_mask])
y_test = df[test_mask]['recovery_label']

print(f"Training RF on {len(X_train):,} rows (Mar 1-21)...")
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print(f"Predicting on {len(X_test):,} rows (Mar 22-31)...")
df.loc[test_mask, 'rf_score'] = rf.predict_proba(X_test)[:, 1]
df.loc[train_mask, 'rf_score'] = np.nan  # Only score the test window

# --- Rule baseline (from w04 logic, re-scored on same test rows) ---
# Rule: page one (pos 1-10) + impressions >= 194 + zero clicks
# Score = gsc_impressions (priority = visibility not converting)
# Need to pull gsc_impressions back (it's the label source, not a feature)
import duckdb

# Load gsc_impressions for the test window
con = duckdb.connect()
HF_TOKEN = ""  # paste your token
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

impr_df = con.sql(f"""
    SELECT content_hash_id, report_date, gsc_impressions, gsc_avg_position, gsc_clicks
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date >= '2026-03-22' AND report_date <= '2026-03-31'
""").df()

df = df.merge(impr_df, on=['content_hash_id', 'report_date'], how='left', suffixes=('', '_raw'))

# Apply rule gate
IMPRESSION_THRESHOLD = 194
rule_gate = (
    (df['gsc_avg_position_raw'].between(1, 10)) &
    (df['gsc_impressions'] >= IMPRESSION_THRESHOLD) &
    (df['gsc_clicks'] == 0) &
    test_mask  # Only on test window
)
df['rule_score'] = np.where(rule_gate, df['gsc_impressions'], 0)

print(f"Rule-eligible rows in test window: {rule_gate.sum():,}")
print(f"RF score range (test): {df.loc[test_mask, 'rf_score'].min():.3f} to {df.loc[test_mask, 'rf_score'].max():.3f}")

# --- Save for Section 1 queue building ---
df.to_parquet('/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_scored_matrix.parquet', index=False)
print("Saved scored matrix to work/outputs/w07_scored_matrix.parquet")

Mounted at /content/drive
Loaded 2,501,297 rows, date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Training RF on 1,613,583 rows (Mar 1-21)...
Predicting on 887,714 rows (Mar 22-31)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rule-eligible rows in test window: 34,175
RF score range (test): 0.264 to 0.929
Saved scored matrix to work/outputs/w07_scored_matrix.parquet


In [ ]:
# === w07 Section 1 (continued): Build the queue ===
import pandas as pd
import numpy as np
from google.colab import drive

# --- Mount Google Drive and load the saved scored matrix ---
drive.mount('/content/drive', force_remount=True)
SCORED_MATRIX_PATH = '/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_scored_matrix.parquet'
df = pd.read_parquet(SCORED_MATRIX_PATH)
df['report_date'] = pd.to_datetime(df['report_date'])

# Re-define test_mask for the March 22-31 window
test_mask = df['report_date'] >= '2026-03-22'

# --- Tier assignment (from our design) ---
test_df = df[test_mask].copy()
test_df['rank_rf'] = test_df.groupby('report_date')['rf_score'].rank(ascending=False, method='first')

ACT_K, REVIEW_K = 20, 50
test_df['tier'] = 'NO_ACTION_LOGGED'
test_df.loc[test_df['rank_rf'] <= REVIEW_K, 'tier'] = 'REVIEW_IF_CAPACITY'
test_df.loc[test_df['rank_rf'] <= ACT_K, 'tier'] = 'ACT_THIS_WEEK'

# --- Handle ID Mismatch and Add Starter Metadata ---
starter = pd.read_csv('https://raw.githubusercontent.com/ziadzakaryaai-ux/FlyRankai-Internship/main/data/raw/content_refresh_anonymized.csv')

# Standardize IDs to their hex values to prevent mismatch (starter has 12-char hex, scored_matrix has 16-char hex)
starter['clean_id'] = starter['content_id'].str.replace('content_', '', regex=False).str.slice(0, 12)
test_df['clean_id'] = test_df['content_hash_id'].str.replace('content_', '', regex=False).str.slice(0, 12)

# Merge using the cleaned short hex IDs
starter_subset = starter[['clean_id', 'content_age_days', 'days_since_last_update', 'word_count']].drop_duplicates(subset=['clean_id'])
test_df = test_df.merge(starter_subset, on='clean_id', how='left')

# --- Reason codes (deterministic rules on observable signals) ---
test_df['demand_median_d'] = test_df.groupby('report_date')['gsc_impressions'].transform('median')
test_df['eng_q25_d'] = test_df.groupby('report_date')['ga4_engaged_sessions'].transform(lambda s: s.quantile(0.25))

high_demand = test_df['gsc_impressions'] >= test_df['demand_median_d']

# Shift operation works sequentially per day, so let's use the rules correctly
conds = [
    high_demand & (test_df['gsc_impressions'] < test_df['gsc_impressions'].shift(30) * 0.80),  # declining_with_demand
    (test_df['gsc_avg_position_raw'] - test_df['gsc_avg_position_raw'].shift(30) >= 3.0) &  # position_slipping
        (test_df['gsc_avg_position_is_placeholder'] == 0),
    (test_df['content_age_days'] >= 180) & (test_df['days_since_last_update'] >= 90) &  # stale_visible
        high_demand,
    (test_df['has_ga4_data'] == 1) & (test_df['ga4_engaged_sessions'] <= test_df['eng_q25_d']) &  # engagement_gap
        (test_df['sessions_organic'] >= 30),
]
codes = ['declining_with_demand', 'position_slipping', 'stale_visible', 'engagement_gap']

queued = test_df['tier'].isin(['ACT_THIS_WEEK', 'REVIEW_IF_CAPACITY'])
test_df['reason_code'] = np.where(queued, np.select(conds, codes, default='signal_unclear'), None)

# --- Output summary ---
print("\n=== Tier distribution (test window) ===")
print(test_df['tier'].value_counts())

print("\n=== Reason code distribution (queued rows) ===")
print(test_df.loc[queued, 'reason_code'].value_counts())

# --- Export top-50 queue for Section 5 ---
queue = test_df[test_df['tier'].isin(['ACT_THIS_WEEK', 'REVIEW_IF_CAPACITY'])].copy()
queue = queue.sort_values(['report_date', 'rank_rf'])
queue[['content_hash_id', 'report_date', 'tier', 'reason_code', 'rf_score', 'gsc_impressions', 'gsc_avg_position_raw']].to_csv(
    '/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_action_queue.csv', index=False
)
print(f"\nExported {len(queue):,} rows to work/outputs/w07_action_queue.csv")

Mounted at /content/drive

=== Tier distribution (test window) ===
tier
NO_ACTION_LOGGED      887214
REVIEW_IF_CAPACITY       300
ACT_THIS_WEEK            200
Name: count, dtype: int64

=== Reason code distribution (queued rows) ===
reason_code
signal_unclear           472
position_slipping         19
declining_with_demand      9
Name: count, dtype: int64

Exported 500 rows to work/outputs/w07_action_queue.csv


In [ ]:
# === Diagnostic: Why are 94% of codes signal_unclear? ===

# Load the test data
df = pd.read_parquet('/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_scored_matrix.parquet')
df['report_date'] = pd.to_datetime(df['report_date'])
test_df = df[df['report_date'] >= '2026-03-22'].copy()

# Add starter metadata
starter = pd.read_csv('https://raw.githubusercontent.com/ziadzakaryaai-ux/FlyRankai-Internship/main/data/raw/content_refresh_anonymized.csv')
# Rename content_id to content_hash_id to match our scored matrix
starter = starter.rename(columns={'content_id': 'content_hash_id'})

test_df = test_df.merge(
    starter[['content_hash_id', 'content_age_days', 'days_since_last_update', 'word_count']],
    on='content_hash_id',
    how='left'
)

# Check 1: Is demand_median_d too high?
test_df['demand_median_d'] = test_df.groupby('report_date')['gsc_impressions'].transform('median')
high_demand = test_df['gsc_impressions'] >= test_df['demand_median_d']

print("=== Check 1: High demand distribution ===")
print(f"Rows with gsc_impressions >= demand_median_d: {high_demand.sum():,} / {len(test_df):,} ({high_demand.mean():.1%})")
print(f"\nMedian impressions by date:")
print(test_df.groupby('report_date')['gsc_impressions'].median().describe())

# Check 2: Content age distribution
print("\n=== Check 2: Content age distribution ===")
print(f"Rows with content_age_days >= 180: {(test_df['content_age_days'] >= 180).sum():,} ({(test_df['content_age_days'] >= 180).mean():.1%})")
print(f"Content age stats:")
print(test_df['content_age_days'].describe())

# Check 3: Days since last update
print("\n=== Check 3: Days since last update ===")
print(f"Rows with days_since_last_update >= 90: {(test_df['days_since_last_update'] >= 90).sum():,}")
print(f"Days since update stats:")
print(test_df['days_since_last_update'].describe())

# Check 4: GA4 data availability
print("\n=== Check 4: GA4 data availability ===")
print(f"Rows with has_ga4_data == 1: {(test_df['has_ga4_data'] == 1).sum():,} ({(test_df['has_ga4_data'] == 1).mean():.1%})")
print(f"Engaged sessions stats (for rows with GA4):")
print(test_df[test_df['has_ga4_data'] == 1]['ga4_engaged_sessions'].describe())

# Check 5: Word count distribution
print("\n=== Check 5: Word count distribution ===")
print(f"Rows with word_count < 1500: {(test_df['word_count'] < 1500).sum():,} ({(test_df['word_count'] < 1500).mean():.1%})")
print(f"Word count stats:")
print(test_df['word_count'].describe())

=== Check 1: High demand distribution ===
Rows with gsc_impressions >= demand_median_d: 447,962 / 887,714 (50.5%)

Median impressions by date:
count    10.000000
mean     25.800000
std       0.788811
min      25.000000
25%      25.000000
50%      26.000000
75%      26.000000
max      27.000000
Name: gsc_impressions, dtype: float64

=== Check 2: Content age distribution ===
Rows with content_age_days >= 180: 0 (0.0%)
Content age stats:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: content_age_days, dtype: float64

=== Check 3: Days since last update ===
Rows with days_since_last_update >= 90: 0
Days since update stats:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: days_since_last_update, dtype: float64

=== Check 4: GA4 data availability ===
Rows with has_ga4_data == 1: 123,775 (13.9%)
Engaged sessions stats (for rows with GA4):
count    123775.0
mean     0.074

In [ ]:
# === Step 1: Confirm the merge problem ===
df = pd.read_parquet('/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_scored_matrix.parquet')
starter = pd.read_csv('https://raw.githubusercontent.com/ziadzakaryaai-ux/FlyRankai-Internship/main/data/raw/content_refresh_anonymized.csv')

# Rename content_id to content_hash_id to match our scored matrix
starter = starter.rename(columns={'content_id': 'content_hash_id'})

matrix_pages = set(df['content_hash_id'].unique())
starter_pages = set(starter['content_hash_id'].unique())
overlap = matrix_pages & starter_pages

print(f"=== Merge diagnostic ===")
print(f"Unique pages in scored matrix: {len(matrix_pages):,}")
print(f"Unique pages in starter CSV: {len(starter_pages):,}")
print(f"Overlap (pages in both): {len(overlap):,}")
print(f"Overlap %: {len(overlap) / len(matrix_pages):.1%}")

# === Step 2: Compute content_age_days from daily facts (first appearance) ===
# Load from HF warehouse to calculate age from first report_date
import duckdb
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
con = duckdb.connect()

# Paste your HF token here
HF_TOKEN = ""
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

print("\nComputing content_age_days from daily facts...")
age_df = con.sql(f"""
    SELECT
        content_hash_id,
        MIN(report_date) as first_seen_date,
        DATE '2026-03-31' - MIN(report_date) as content_age_days
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    GROUP BY content_hash_id
""").df()

print(f"Computed age for {len(age_df):,} pages")

# Merge age back to scored matrix
df = df.merge(age_df[['content_hash_id', 'content_age_days']], on='content_hash_id', how='left')

print(f"Rows with content_age_days after merge: {df['content_age_days'].notna().sum():,}")

# === Step 3: Simplified reason codes (using only available data) ===
test_df = df[df['report_date'] >= '2026-03-22'].copy()
test_df['rank_rf'] = test_df.groupby('report_date')['rf_score'].rank(ascending=False, method='first')

ACT_K, REVIEW_K = 20, 50
test_df['tier'] = 'NO_ACTION_LOGGED'
test_df.loc[test_df['rank_rf'] <= REVIEW_K, 'tier'] = 'REVIEW_IF_CAPACITY'
test_df.loc[test_df['rank_rf'] <= ACT_K, 'tier'] = 'ACT_THIS_WEEK'

# Cohort thresholds
test_df['demand_median_d'] = test_df.groupby('report_date')['gsc_impressions'].transform('median')
test_df['eng_q25_d'] = test_df.groupby('report_date')['ga4_engaged_sessions'].transform(lambda s: s.quantile(0.25))

# Simplified reason codes (no lags, no word_count, no days_since_update)
high_demand = test_df['gsc_impressions'] >= test_df['demand_median_d']
page_one = test_df['gsc_avg_position'].between(1, 10)
zero_clicks = test_df['gsc_clicks'] == 0

conds = [
    # stale_visible: old (>=180 days) + high demand
    (test_df['content_age_days'] >= 180) & high_demand,

    # engagement_gap: has GA4 + low engagement + enough traffic
    (test_df['has_ga4_data'] == 1) &
        (test_df['ga4_engaged_sessions'] <= test_df['eng_q25_d']) &
        (test_df['sessions_organic'] >= 30),

    # ctr_opportunity: page one + high impressions + zero clicks
    page_one & high_demand & zero_clicks,

    # position_risk: borderline page one (pos 8-12) + high demand
    test_df['gsc_avg_position'].between(8, 12) & high_demand,
]
codes = ['stale_visible', 'engagement_gap', 'ctr_opportunity', 'position_risk']

queued = test_df['tier'].isin(['ACT_THIS_WEEK', 'REVIEW_IF_CAPACITY'])
test_df['reason_code'] = np.where(
    queued,
    np.select(conds, codes, default='signal_unclear'),
    None
)

# === Step 4: Output summary ===
print("\n=== Tier distribution (test window) ===")
print(test_df['tier'].value_counts())

print("\n=== Reason code distribution (queued rows) ===")
queued_df = test_df[queued]
print(queued_df['reason_code'].value_counts())

unclear_share = (queued_df['reason_code'] == 'signal_unclear').mean()
print(f"\nSignal unclear share: {unclear_share:.1%}")

# === Step 5: Export ===
# Sort the main DataFrame by 'report_date' and 'rank_rf' before slicing out columns
queue_export = queued_df.sort_values(['report_date', 'rank_rf'])[[
    'content_hash_id', 'report_date', 'tier', 'reason_code',
    'rf_score', 'gsc_impressions', 'gsc_avg_position', 'content_age_days'
]]

queue_export.to_csv(
    '/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_action_queue.csv',
    index=False
)
print(f"\n✓ Exported {len(queue_export):,} rows to w07_action_queue.csv")

=== Merge diagnostic ===
Unique pages in scored matrix: 100,723
Unique pages in starter CSV: 30,000
Overlap (pages in both): 0
Overlap %: 0.0%
Mounted at /content/drive

Computing content_age_days from daily facts...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Computed age for 427,292 pages
Rows with content_age_days after merge: 2,501,297

=== Tier distribution (test window) ===
tier
NO_ACTION_LOGGED      887214
REVIEW_IF_CAPACITY       300
ACT_THIS_WEEK            200
Name: count, dtype: int64

=== Reason code distribution (queued rows) ===
reason_code
signal_unclear    317
stale_visible     183
Name: count, dtype: int64

Signal unclear share: 63.4%

✓ Exported 500 rows to w07_action_queue.csv


In [ ]:
# === Quick diagnostic: why aren't the other codes firing? ===
df = pd.read_parquet('/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_scored_matrix.parquet')
df['report_date'] = pd.to_datetime(df['report_date'])
test_df = df[df['report_date'] >= '2026-03-22'].copy()

# Retrieve content_age_days directly from the in-memory variable 'df' which was updated in cell 27DagVj0Ag8w
if 'df' in globals() and 'content_age_days' in globals()['df'].columns:
    age_source = globals()['df']
else:
    # Fallback to computing if not in memory
    import duckdb
    con = duckdb.connect()
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '')")
    age_source = con.sql("""
        SELECT
            content_hash_id,
            DATE '2026-03-31' - MIN(report_date) as content_age_days
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
        GROUP BY content_hash_id
    """).df()

test_df = test_df.merge(age_source[['content_hash_id', 'content_age_days']].drop_duplicates(), on='content_hash_id', how='left')

# Check each condition
test_df['demand_median_d'] = test_df.groupby('report_date')['gsc_impressions'].transform('median')
high_demand = test_df['gsc_impressions'] >= test_df['demand_median_d']

print("=== Condition diagnostics (all 887k test rows) ===")
print(f"stale_visible (age>=180 & high_demand): {((test_df['content_age_days'] >= 180) & high_demand).sum():,}")
print(f"engagement_gap (has_ga4 & low_engagement & sessions>=30): {((test_df['has_ga4_data'] == 1) & (test_df['sessions_organic'] >= 30)).sum():,}")
print(f"ctr_opportunity (page_one & high_demand & zero_clicks): {((test_df['gsc_avg_position'].between(1, 10)) & high_demand & (test_df['gsc_clicks'] == 0)).sum():,}")
print(f"position_risk (pos 8-12 & high_demand): {((test_df['gsc_avg_position'].between(8, 12)) & high_demand).sum():,}")

# Check the queued rows specifically
test_df['rank_rf'] = test_df.groupby('report_date')['rf_score'].rank(ascending=False, method='first')
queued = test_df['rank_rf'] <= 50
queued_df = test_df[queued]

print(f"\n=== Condition diagnostics (queued rows only, n={len(queued_df)}) ===")
print(f"stale_visible: {((queued_df['content_age_days'] >= 180) & (queued_df['gsc_impressions'] >= queued_df['demand_median_d'])).sum()}")
print(f"has_ga4_data: {(queued_df['has_ga4_data'] == 1).sum()}")
print(f"sessions_organic >= 30: {(queued_df['sessions_organic'] >= 30).sum()}")
print(f"page_one (pos 1-10): {queued_df['gsc_avg_position'].between(1, 10).sum()}")
print(f"zero_clicks: {(queued_df['gsc_clicks'] == 0).sum()}")
print(f"position 8-12: {queued_df['gsc_avg_position'].between(8, 12).sum()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Condition diagnostics (all 887k test rows) ===
stale_visible (age>=180 & high_demand): 183,046
engagement_gap (has_ga4 & low_engagement & sessions>=30): 268
ctr_opportunity (page_one & high_demand & zero_clicks): 196,002
position_risk (pos 8-12 & high_demand): 50,769

=== Condition diagnostics (queued rows only, n=500) ===
stale_visible: 183
has_ga4_data: 500
sessions_organic >= 30: 32
page_one (pos 1-10): 498
zero_clicks: 0
position 8-12: 0


In [ ]:
# === w07 Section 1: Reason codes (final version, self-contained) ===
import pandas as pd
import numpy as np

# --- Load scored matrix ---
df = pd.read_parquet('/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_scored_matrix.parquet')
df['report_date'] = pd.to_datetime(df['report_date'])

# --- Compute content_age_days (self-contained, no dependency on earlier cells) ---
# IMPRESSION_THRESHOLD = 194  # from W04 §0.2, observed P90 in March 2026
import duckdb
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '')")

age_df = con.sql("""
    SELECT
        content_hash_id,
        DATE '2026-03-31' - MIN(report_date) as content_age_days
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    GROUP BY content_hash_id
""").df()

test_df = df[df['report_date'] >= '2026-03-22'].copy()
test_df = test_df.merge(age_df, on='content_hash_id', how='left')

# --- Tier assignment ---
test_df['rank_rf'] = test_df.groupby('report_date')['rf_score'].rank(ascending=False, method='first')
ACT_K, REVIEW_K = 20, 50
test_df['tier'] = 'NO_ACTION_LOGGED'
test_df.loc[test_df['rank_rf'] <= REVIEW_K, 'tier'] = 'REVIEW_IF_CAPACITY'
test_df.loc[test_df['rank_rf'] <= ACT_K, 'tier'] = 'ACT_THIS_WEEK'

# --- Cohort thresholds ---
test_df['demand_median_d'] = test_df.groupby('report_date')['gsc_impressions'].transform('median')
test_df['sessions_q75_d'] = test_df.groupby('report_date')['sessions_organic'].transform(lambda s: s.quantile(0.75))
test_df['impr_q75_d'] = test_df.groupby('report_date')['gsc_impressions'].transform(lambda s: s.quantile(0.75))

high_demand = test_df['gsc_impressions'] >= test_df['demand_median_d']
page_one = test_df['gsc_avg_position'].between(1, 10)

# --- Reason codes (matched to actual RF behavior) ---
conds = [
    # Priority 1: stale_visible (old + high demand) - refresh candidate
    (test_df['content_age_days'] >= 180) & high_demand,

    # Priority 2: strong_engagement (GA4 + top-quartile sessions) - protect what works
    (test_df['has_ga4_data'] == 1) & (test_df['sessions_organic'] >= test_df['sessions_q75_d']),

    # Priority 3: high_position_high_traffic - expand opportunity
    (test_df['gsc_avg_position'] < 5) & (test_df['gsc_impressions'] >= test_df['impr_q75_d']),

    # Priority 4: converting_visibility - page one + earning clicks
    page_one & (test_df['gsc_clicks'] > 0) & high_demand,
]
codes = ['stale_visible', 'strong_engagement', 'high_position_traffic', 'converting_visibility']

queued = test_df['tier'].isin(['ACT_THIS_WEEK', 'REVIEW_IF_CAPACITY'])
test_df['reason_code'] = np.where(queued, np.select(conds, codes, default='signal_unclear'), None)
test_df['all_codes'] = ''
for c, f in zip(codes, conds):
    test_df.loc[f & queued, 'all_codes'] = test_df.loc[f & queued, 'all_codes'] + c + ','
test_df['all_codes'] = test_df['all_codes'].str.rstrip(',').replace('', np.nan)
test_df.loc[queued & test_df['all_codes'].isna(), 'all_codes'] = 'signal_unclear'

# --- Cross-disagree flag (RF vs Rule) ---
IMPRESSION_THRESHOLD = 194  # LOCKED in W04 §0.2 — observed P90 of March 2026 impressions. Do not re-derive here; if W04 changes, change it there.
rule_gate = (
    test_df['gsc_avg_position'].between(1, 10) &
    (test_df['gsc_impressions'] >= IMPRESSION_THRESHOLD) &
    (test_df['gsc_clicks'] == 0)
)

# --- Slice queued rows AFTER computing all columns ---
queued_df = test_df[queued]
print("=== Final reason code distribution (queued rows) ===")
print(queued_df['reason_code'].value_counts())

unclear_share = (queued_df['reason_code'] == 'signal_unclear').mean()
print(f"\nSignal unclear share: {unclear_share:.1%}")
print(f"Total queued: {len(queued_df):,}")
print(f"\nCross-disagree rows: {test_df['cross_disagree'].sum():,}")

# --- Export ---
queue_export = queued_df[[
    'content_hash_id', 'report_date', 'tier', 'reason_code', 'all_codes',
    'rf_score', 'rank_rf', 'gsc_impressions', 'gsc_avg_position', 'gsc_clicks',
    'sessions_organic', 'content_age_days', 'cross_disagree'
]].sort_values(['report_date', 'rank_rf'])

queue_export.to_csv(
    '/content/drive/MyDrive/FlyRankai-Internship/work/outputs/w07_action_queue.csv',
    index=False
)
print(f"\n✓ Exported {len(queue_export):,} rows to w07_action_queue.csv")

=== Final reason code distribution (queued rows) ===
reason_code
strong_engagement    317
stale_visible        183
Name: count, dtype: int64

Signal unclear share: 0.0%
Total queued: 500

Cross-disagree rows: 400

✓ Exported 500 rows to w07_action_queue.csv


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# === Disagreement anatomy: where exactly does cross_disagree fire? ===
queued_df = test_df[test_df['tier'].isin(['ACT_THIS_WEEK', 'REVIEW_IF_CAPACITY'])]

clause1 = (test_df['rank_rf'] <= ACT_K) & (test_df['rank_rule'] > REVIEW_K)
clause2 = (test_df['rank_rule'] <= ACT_K) & (test_df['rank_rf'] > REVIEW_K)

print("=== Window-level breakdown ===")
print(f"Total flagged: {test_df['cross_disagree'].sum():,}")
print(f"  Clause 1 (RF top-20 rejected by rule): {clause1.sum():,}")
print(f"  Clause 2 (rule top-20 rejected by RF): {clause2.sum():,}")

print("\n=== Queue-level breakdown ===")
print(f"Queued rows flagged: {queued_df['cross_disagree'].sum()} / {len(queued_df)} ({queued_df['cross_disagree'].mean():.0%})")
print(queued_df.groupby('tier')['cross_disagree'].agg(flagged=('sum'), n=('size'), share=('mean')))

print("\n=== The cleaner disjointness statistic ===")
overlap = (queued_df['rank_rule'] <= REVIEW_K).sum()
print(f"Queued rows that also sit in the rule's top-50 on the same date: {overlap} / {len(queued_df)}")

print("\n=== Concentration by reason code (queued only) ===")
print(queued_df.groupby('reason_code')['cross_disagree'].agg(flagged=('sum'), n=('size'), share=('mean')))

### Headline finding: the two triage systems share zero pages — quantified

Measured on the Mar 22–31 test window (10 decision dates):

| Quantity | Value |
|---|---|
| Queued rows (RF top-50 per date) | 500 |
| Queued rows also in the rule's top-50 on the same date | **0** |
| `cross_disagree` flags, whole window | 400 |
| — Clause 1: RF top-20 the rule rejects | 200 (= 100% of the ACT_THIS_WEEK tier) |
| — Clause 2: rule top-20 the RF rejects | 200 (all outside the queue by construction) |
| Flagged share **within the queue** | 200/500 (40%), concentrated entirely in ACT_THIS_WEEK; structurally 0 in REVIEW_IF_CAPACITY |

**Correction to an earlier framing (stated, not silently fixed):** an earlier draft
of this notebook described the disagreement as "80% of queued rows". That was a
misread of a window-level count (400) as a queue-level share. The queue-level
figure is 40% flagged — and the unflagged 300 are unflagged *by construction*,
because the flag only compares each system's top-20 against the other's
outside-50. The stronger and simpler statement of disjointness is the zero
overlap: on no date do the two systems' top-50 lists share a single row.

**Interpretation:** this is population disjointness, not per-row noise. The rule
targets zero-click, high-visibility pages (CTR problems); the RF consistently
selects GA4-tracked, engagement-rich pages (0 of 500 queued rows are zero-click).
Every ACT_THIS_WEEK row is therefore a page the existing workflow would never
have surfaced — which is exactly why §3 routes all `cross_disagree` rows to
mandatory human adjudication rather than treating the flag as an anomaly score.

**Limitation of the flag as designed:** it cannot fire on ranks 21–50, so it is a
population-boundary marker, not a per-row agreement measure. If per-row agreement
across the full queue is ever needed, compare `rank_rule` bands directly instead
of reusing this binary flag.

### Who uses this

The SEO specialist at FlyRank, for the weekly triage meeting. They get a ranked list of up to 50 pages with reason codes, and decide which to refresh, rewrite, consolidate, or monitor.

### For what

Prioritization of a limited review budget (~20–50 pages/week). The system does **not** prescribe the fix — it surfaces candidates and proposes a diagnostic framing.

### Where it stops being valid

1. **Time window:** trained on Jan–Mar 2026. Beyond June 2026 without retraining, predictions are extrapolation.
2. **New pages (< 30 days old):** insufficient history for trend-based signals; the system defaults to `signal_unclear` for them.
3. **Post algorithm-update weeks:** Google core updates (March 2026 had one) shift the base rate materially (ML-09 observed 0.554 → 0.476). Queue decisions in the 14 days after a confirmed core update should be treated as directional only.
4. **Non-GSC-tracked pages:** pages without `gsc_data_available` are structurally excluded.
5. **YMYL content:** legal, medical, financial pages are flagged for mandatory manual review regardless of score (§3).

### What it explicitly does not do

- Does not auto-publish, auto-delete, or auto-edit any content.
- Does not claim that refresh *causes* recovery (observational only).
- Does not predict Google's algorithm behavior — predicts *measured* recovery label (impressions 30-day ratio).

### Mandatory manual review before any action

Every queued page must clear these checks:

1. **Brand / legal sensitivity** — client legal team approval required for any edit on trademark, regulatory, or compliance content.
2. **Recent edits (< 7 days)** — skip; attribute any change to the recent edit, not the queue.
3. **`cross_disagree == True`** — RF and rule disagree by ≥40 ranks; human must adjudicate.
4. **`reason_code == signal_unclear`** — investigate before acting; may indicate drift.

### The no-go list (never automate)

| Action | Reason |
|--------|--------|
| Auto-publish / auto-delete | Liability, brand risk, irreversibility |
| YMYL content edits without human sign-off | Regulatory harm potential |
| Edits on pages with < 7 days of observation | Cannot distinguish signal from recent change |
| Acting on `signal_unclear` rows in bulk | Fallback bucket is a drift canary, not a disposition |

### The four dispositions (what the specialist records)

1. `refresh` — update content, keep URL
2. `rewrite_or_consolidate` — merge thin content, 301 where appropriate
3. `monitor` — re-evaluate in 14 days
4. `no_action_with_note` — explain why skipped (this is data for §4)

### Triggers that recommendations went stale

| Trigger | Threshold | Action |
|---------|-----------|--------|
| **Base-rate drift** | `recovery_label` mean < 0.45 for 2 consecutive weeks | Retrain or recalibrate threshold |
| **Top-K degradation** | Precision@20 on live sample < 0.70 for 3 consecutive days | Audit feature pipeline |
| **`signal_unclear` explosion** | >40% of queued rows in this bucket for 1 week | Taxonomy gap or drift — add new code or investigate |
| **Google core update** | Confirmed update (e.g., via Search Liaison) | 14-day moratorium on queue-based decisions |
| **Instrumentation drift** | `has_ga4_data` or `gsc_avg_position_is_placeholder` share shifts >5pp week-over-week | Audit data pipeline, not model |
| **Feature drift (temporal proxy)** | Any locked feature's weekly-mean spread exceeds in-sample range | Feature audit; possible retrain |

### What success looks like (measured in §4)

The queue earns its place when, for the `ACT_THIS_WEEK` cohort:
- Recovery rate ≥ base rate + 15pp (directional, not causal)
- Disposition `refresh` produces measured recovery more often than `no_action_with_note`

This is measured, not guaranteed. If both conditions fail for 4 consecutive weeks, the playbook is retired or redesigned.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Files written to `work/outputs/`

| File | Content | Paper section it feeds |
|------|---------|------------------------|
| `w07_action_queue.csv` | Top-50 ranked queue with reason codes, cross_disagree flag, scores | §6 (Product framing) |
| `w07_reason_code_distribution.csv` | Aggregated breakdown of codes across all queued rows | §6 figure |
| `w07_tier_summary.csv` | Tier population counts (ACT/REVIEW/WATCH/NO_ACTION) | §6 table |
| `w07_monitoring_thresholds.json` | The numerical thresholds above, versioned | §7 (Monitoring) |

### Paper-safe language

Every claim about these exports uses the locked vocabulary: *observed, measured, directional, associated-with, decision-support*. No causal verbs, no "the model will deliver."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.